## Serverless API

In the Hugging Face ecosystem, there is a convenient feature called Serverless API that allows you to easily run inference on many models. There's no installation or deployment required.

To run this notebook, **you need a Hugging Face token** that you can get from https://hf.co/settings/tokens. A "Read" token type is sufficient.
- If you are running this notebook on Google Colab, you can set it up in the "settings" tab under "secrets". Make sure to call it "HF_TOKEN" and restart the session to load the environment variable (Runtime -> Restart session).
- If you are running this notebook locally, you can set it up as an [environment variable](https://huggingface.co/docs/huggingface_hub/en/package_reference/environment_variables). Make sure you restart the kernel after installing or updating huggingface_hub. You can update huggingface_hub by modifying the above `!pip install -q huggingface_hub -U`

In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
hf_token=os.getenv("HF_TOKEN")

client=InferenceClient(model='moonshotai/Kimi-K2.5', token=hf_token)

We use the `chat` method since it is a convenient and reliable way to apply chat templates:

In [5]:
output=client.chat.completions.create(messages=[{'role':'user', 'content':'Adolf Hitler died in'}], 
                                      stream=False, max_tokens=1024, 
                                      extra_body={'thinking':{'type':'disabled'}})

# Thinking is disabled here, so llm will give the output it generates on its first try and block the usage of thinking setting.
# What is thinking?: https://platform.claude.com/docs/en/build-with-claude/thinking

print(output.choices[0].message.content)

Adolf Hitler died on **April 30, 1945**, in his Führerbunker in Berlin. He committed suicide by gunshot, alongside his wife Eva Braun, who took cyanide. Their bodies were subsequently burned to prevent capture by Soviet troops who were closing in on the city.


The chat method is the RECOMMENDED method to use in order to ensure a **smooth transition between models**.

## Dummy Agent

In the previous sections, we saw that the **core of an agent library is to append information in the system prompt**.

This system prompt is a bit more complex than the one we saw earlier, but it already contains:

1. **Information about the tools**
2. **Cycle instructions** (Thought → Action → Observation)

In [ ]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """